In [2]:
# wav2vec2_finetune_gpu_safe.py
import os
import math
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    Wav2Vec2ForSequenceClassification,
    Wav2Vec2FeatureExtractor,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
from tqdm import tqdm
import torchaudio

# -------------------------
# Config (tune these)
# -------------------------
CSV_PATH = "voxpopuli_with_mfcc.csv"   # <<--- update if needed (must contain audio_path and lang)
CHECKPOINT_DIR = "av2vec2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TARGET_SR = 16000
MAX_AUDIO_SEC = 8
MAX_AUDIO_SAMPLES = int(TARGET_SR * MAX_AUDIO_SEC)

BATCH_SIZE = 8
EPOCHS = 20
UNFREEZE_LAST_N = 2
BASE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 3
SAVE_EVERY_N_STEPS = 200

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

# Choose num_workers safely (0 on Windows)
import platform, multiprocessing
if platform.system() == "Windows":
    NUM_WORKERS = 0
else:
    NUM_WORKERS = max(0, min(8, (multiprocessing.cpu_count() - 1)))

PIN_MEMORY = True  # safe now since DataLoader returns CPU tensors

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    try:
        print("GPU:", torch.cuda.get_device_name(0))
    except Exception:
        pass
print(f"DataLoader num_workers={NUM_WORKERS}  pin_memory={PIN_MEMORY}")

# -------------------------
# Load CSV, labels
# -------------------------
df = pd.read_csv(CSV_PATH)

# Ensure your CSV contains a column with path to audio (audio_path) and language label (lang)
AUDIO_COL = "audio_path"
LABEL_COL = "lang"
if AUDIO_COL not in df.columns:
    raise RuntimeError(f"CSV must contain a column named '{AUDIO_COL}' with audio file paths.")
if LABEL_COL not in df.columns:
    raise RuntimeError(f"CSV must contain a column named '{LABEL_COL}' with language labels.")

le = LabelEncoder()
df['label_id'] = le.fit_transform(df[LABEL_COL])
num_labels = len(le.classes_)
label2id = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}
id2label = {int(v): str(k) for k, v in zip(le.classes_, le.transform(le.classes_))}

train_df, val_df = train_test_split(
    df, test_size=0.12, random_state=42, stratify=df['label_id']
)

print("\n===== DATA INFO =====")
print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches per epoch (train): {math.ceil(len(train_df) / BATCH_SIZE)}")
print(f"Total epochs: {EPOCHS}")

# -------------------------
# Audio loader using torchaudio (worker-friendly)
# -------------------------
def load_audio_np(path, target_sr=TARGET_SR, max_samples=MAX_AUDIO_SAMPLES):
    # returns 1D float32 numpy array, length == max_samples (padded/truncated)
    waveform, sr = torchaudio.load(path)  # shape [channels, samples], float32
    if waveform.size(0) > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=target_sr)
    if waveform.numel() > max_samples:
        waveform = waveform[:max_samples]
    elif waveform.numel() < max_samples:
        pad = torch.zeros(max_samples - waveform.numel(), dtype=waveform.dtype)
        waveform = torch.cat((waveform, pad), dim=0)
    return waveform.numpy().astype(np.float32)

# -------------------------
# Dataset + collate
# -------------------------
class AudioDataset(Dataset):
    def __init__(self, df, audio_col=AUDIO_COL):
        self.paths = df[audio_col].tolist()
        self.labels = df['label_id'].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        audio = load_audio_np(path)
        label = int(self.labels[idx])
        # return CPU types only (numpy + int)
        return {"audio": audio, "label": label}

def collate_fn(batch):
    # **Return CPU-only structures** (no tensors moved to GPU here)
    audios = [item['audio'] for item in batch]         # list of numpy arrays (float32)
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)  # CPU tensor
    return {"input_values": audios, "labels": labels}

train_loader = DataLoader(
    AudioDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    AudioDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

# -------------------------
# Model + feature extractor
# -------------------------
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

# Freeze encoder then unfreeze last N layers
for p in model.wav2vec2.parameters():
    p.requires_grad = False

try:
    encoder_layers = model.wav2vec2.encoder.layers
    n_layers = len(encoder_layers)
    for i in range(max(0, n_layers - UNFREEZE_LAST_N), n_layers):
        for p in encoder_layers[i].parameters():
            p.requires_grad = True
    print(f"\nUnfroze last {UNFREEZE_LAST_N}/{n_layers} encoder layers.")
except Exception as e:
    print("Partial unfreeze failed:", e)

# Always train classifier (+ projector if exists)
for p in model.classifier.parameters():
    p.requires_grad = True
if hasattr(model, "projector"):
    for p in model.projector.parameters():
        p.requires_grad = True

model.to(DEVICE)

# -------------------------
# Optimizer + scheduler
# -------------------------
encoder_params = []
head_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "wav2vec2.encoder" in name:
        encoder_params.append(param)
    else:
        head_params.append(param)

optimizer = AdamW([
    {"params": encoder_params, "lr": BASE_LR},
    {"params": head_params, "lr": HEAD_LR}
], weight_decay=WEIGHT_DECAY)

total_steps = math.ceil(len(train_loader) * EPOCHS)
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# -------------------------
# Evaluation (GPU-safe)
# -------------------------
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    nbatches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="🔍 Validating", leave=False):
            # feature_extractor returns CPU tensors -> move to device AFTER creation
            batch_inputs = feature_extractor(
                batch["input_values"],
                sampling_rate=TARGET_SR,
                padding="longest",
                return_tensors="pt",
                return_attention_mask=True
            )
            input_values = batch_inputs["input_values"].to(DEVICE, non_blocking=True)
            attention_mask = batch_inputs["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            outputs = model(
                input_values=input_values,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            nbatches += 1

            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, nbatches)
    acc = accuracy_score(all_labels, all_preds) if len(all_labels) > 0 else 0.0
    return avg_loss, acc, all_preds, all_labels

# -------------------------
# Resume from checkpoint (if any)
# -------------------------
best_val_loss = float("inf")
best_epoch = -1
patience_counter = 0
start_epoch = 1
global_step = 0

ckpt_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("checkpoint_step")]
if ckpt_files:
    latest_ckpt = max(ckpt_files, key=lambda x: int(x.split("step")[1].split(".")[0]))
    ckpt_path = os.path.join(CHECKPOINT_DIR, latest_ckpt)
    print("Loading checkpoint:", ckpt_path)
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if USE_AMP and "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint.get("epoch", 1)
    global_step = checkpoint.get("global_step", 0)
    best_val_loss = checkpoint.get("best_val_loss", best_val_loss)
    print(f" Resumed epoch {start_epoch}  global_step {global_step}")

# -------------------------
# Training loop (GPU-safe)
# -------------------------
print("\n🚀 Starting training on device:", DEVICE)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    print(f"\n================== EPOCH {epoch}/{EPOCHS} ==================")
    progress = tqdm(enumerate(train_loader), total=len(train_loader))

    for step, batch in progress:
        global_step += 1

        # Build inputs on CPU, then move to device
        batch_inputs = feature_extractor(
            batch["input_values"],
            sampling_rate=TARGET_SR,
            padding="longest",
            return_tensors="pt",
            return_attention_mask=True
        )
        input_values = batch_inputs["input_values"].to(DEVICE, non_blocking=True)
        attention_mask = batch_inputs["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(
                input_values=input_values,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        avg_loss = running_loss / (step + 1)
        current_lr = scheduler.get_last_lr()[0]

        progress.set_description(f"Epoch {epoch}/{EPOCHS} | Step {step+1}/{len(train_loader)}")
        progress.set_postfix({
            "loss": f"{loss.item():.4f}",
            "avg_loss": f"{avg_loss:.4f}",
            "lr": f"{current_lr:.2e}"
        })

        # Save checkpoint periodically
        if global_step % SAVE_EVERY_N_STEPS == 0:
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_step{global_step}.pt")
            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict() if USE_AMP else None,
                "best_val_loss": best_val_loss
            }, ckpt_path)
            print(f"\n💾 Saved checkpoint at step {global_step}")

    # Validation
    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader)

    print(f"\n📊 EPOCH {epoch} SUMMARY:")
    print(f"   Train Loss: {running_loss/len(train_loader):.4f}")
    print(f"   Val Loss:   {val_loss:.4f}")
    print(f"   Val Acc:    {val_acc:.4f}")

    # Best model + early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"best_model_epoch{epoch}.pt"))
        print("✅ New best model saved.")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⏳ No improvement. Patience {patience_counter}/{PATIENCE}")

    # Save epoch checkpoint for resume
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_step{global_step}.pt")
    torch.save({
        "epoch": epoch + 1,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if USE_AMP else None,
        "best_val_loss": best_val_loss
    }, ckpt_path)

    if patience_counter >= PATIENCE:
        print("🛑 Early stopping triggered.")
        break

# -------------------------
# Final evaluation
# -------------------------
print(f"\n🏁 Training complete! Best epoch: {best_epoch} | Best Val Loss: {best_val_loss:.4f}")

if best_epoch > 0:
    best_model_path = os.path.join(CHECKPOINT_DIR, f"best_model_epoch{best_epoch}.pt")
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    model.to(DEVICE)

    val_loss, val_acc, val_preds, val_labels = evaluate(model, val_loader)

    print("\n✅ Final Best Model Evaluation:")
    print(f"Val Loss: {val_loss}")
    print(f"Val Acc:  {val_acc}")
    print(classification_report(val_labels, val_preds, target_names=list(le.classes_)))
    print("Confusion matrix:\n", confusion_matrix(val_labels, val_preds))


Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Ti
DataLoader num_workers=0  pin_memory=True

===== DATA INFO =====
Train samples: 13200
Validation samples: 1800
Batch size: 8
Batches per epoch (train): 1650
Total epochs: 20


c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Unfroze last 2/12 encoder layers.


C:\Users\Pc\AppData\Local\Temp\ipykernel_19940\1366995590.py:214: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)



🚀 Starting training on device: cuda

================== EPOCH 1/20 ==================


...


💾 Saved checkpoint at step 8200


Epoch 5/20 | Step 1650/1650: 100%|██████████| 1650/1650 [18:24<00:00,  1.49it/s, loss=1.0988, avg_loss=1.0983, lr=7.98e-06]



📊 EPOCH 5 SUMMARY:
   Train Loss: 1.0983
   Val Loss:   1.0985
   Val Acc:    0.3261
⏳ No improvement. Patience 3/3
🛑 Early stopping triggered.

🏁 Training complete! Best epoch: 2 | Best Val Loss: 1.0985



✅ Final Best Model Evaluation:
Val Loss: 1.0985001129574246
Val Acc:  0.3372222222222222


TypeError: object of type 'numpy.int64' has no len()